In [1]:
import pandas as pd
from collections import defaultdict
from collections import Counter

In [2]:
#Load CBM data from Rennison https://doi.org/10.1002/cssc.202500468
df = pd.read_excel('Supplementarytable1_Rennison_highthroughput.xlsx', sheet_name = 'CBM_sequences_selected', header = 0)

#Keep only CBM2 proteins
df = df[df['CBM family'] == 'CBM2']

df

,Header,Protein ID,CBM family,Sequence
0,AAA75477|Acidothermus cellulolyticus 11B|465-562,APR1,CBM2,GCTASYQVNSDWGNGFTVTVAVTNSGSVATKTWTVSWTFGGNQTIT...
1,AAC06196|Actinomyces sp. 40|380-481,APR2,CBM2,GAASGGLTATAAIRNQWNSGATYAITVSNASGSKHEGAWQVTFDLD...
2,CAA11769|Amycolatopsis orientalis|393-495,APR3,CBM2,GDCSVAYRVDSQWGNGFTASVTVTNNGTTAIDGWSLRWTFENSQQV...
3,CAA35574|Butyrivibrio fibrisolvens H17C|451-547,APR4,CBM2,GVSGALKAEYTINNWGSGYQVLIKVKNDSASRVDGWTLKISKSEVK...
4,AAC36898|Cellulomonas fimi ATCC 484|769-872,APR5,CBM2,GGCTVKYSASSWNTGFTGTVEVKNNGTAALNGWTLGFSFADGQKVS...
...,...,...,...,...
904,ACY98980|Thermomonospora curvata DSM 43183|97-197,APR905,CBM2,GVLIAFRVARRLPDGFTGTVTIANRTSRPIDGWLLGFKFPKAEVLT...
905,ADJ46832|Amycolatopsis mediterranei U32|1039-1148,APR906,CBM2,GVSVTITKTADTGSTYTDTATITNNSGSAVNGWNLQFDLVGRMTVT...
906,ADJ46833|Amycolatopsis mediterranei U32|511-612,APR907,CBM2,GATVTVTKTADTGTGTYTDQAVITNNSGADVHGWNVQFDLSGRMTV...
907,CCA94939|uncultured eukaryote|278-369,APR908,CBM2,GCSAVITQSLGQTWNGGGLINIVIINTGTSPISQVSFNLAQDGIAT...


In [3]:
#Load assay results from Rennison https://doi.org/10.1002/cssc.202500468
df2 = pd.read_excel('Supplementarytable1_Rennison_highthroughput.xlsx', sheet_name = 'Holdup_results', header = 0)

#Keep only CBM2 proteins
df2 = df2[df2['Protein ID'].isin(df['Protein ID'])]
df2

,Protein ID,Header,Avicel,Chitin,Starch,PET,LDPE,PS,Unnamed: 8,Unnamed: 9,Unnamed: 10
0,APR97,ABM33954|Acidovorax citrulli AAC00-1|276-351,NB,0.236303,NB,NB,NB,NB,NaN,NB,Binding below EGFP levels on that substrate
1,APR98,ABD80910|Saccharophagus degradans 2-40|816-914,NB,0.173267,NB,NB,NB,0.22,NaN,BD,Data discarded for reasons of data quality
2,APR99,CAJ57849|Cellulomonas flavigena CDBB-531|252-332,0.457947,0.18935,0.120365,NB,BD,NB,NaN,NaN,NaN
3,APR101,ABM79533|Prochlorococcus marinus str. MIT 9303...,NB,0.111891,NB,NB,NB,NB,NaN,NaN,NaN
4,APR102,ABP54735|Salinispora tropica CNB-440|33-135,0.3,0.409836,NB,0.396884,BD,NB,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
524,APR904,BAH85844|Mizuhopecten yessoensis|21-115,BD,0.266089,0.106986,NB,BD,0.312427,NaN,NaN,NaN
525,APR905,ACY98980|Thermomonospora curvata DSM 43183|97-197,NB,0.519311,NB,NB,NB,0.190924,NaN,NaN,NaN
526,APR906,ADJ46832|Amycolatopsis mediterranei U32|1039-1148,NB,0.282393,NB,NB,NB,0.264321,NaN,NaN,NaN
527,APR907,ADJ46833|Amycolatopsis mediterranei U32|511-612,NB,NB,NB,NB,0.171275,NB,NaN,NaN,NaN


In [4]:
# IDs of proteins binding to specific substrates 

ids_cel = df2.loc[(df2['Avicel'] != 'NB')  & (df2['Avicel'] != 'BD'), 'Protein ID'].tolist() #Cellulose-binding
ids_chi = df2.loc[(df2['Chitin'] != 'NB')  & (df2['Chitin'] != 'BD'), 'Protein ID'].tolist() #Chitin-binding
ids_sta = df2.loc[(df2['Starch'] != 'NB')  & (df2['Starch'] != 'BD'), 'Protein ID'].tolist() #Starch-binding
ids_PET = df2.loc[(df2['PET'] != 'NB')  & (df2['PET'] != 'BD'), 'Protein ID'].tolist()       #PET-binding
ids_LDPE = df2.loc[(df2['LDPE'] != 'NB')  & (df2['LDPE'] != 'BD'), 'Protein ID'].tolist()    #LDPE-binding
ids_PS = df2.loc[(df2['PS'] != 'NB')  & (df2['PS'] != 'BD'), 'Protein ID'].tolist()          #PS-binding




In [7]:
# Find IDs of specific vs promiscuous proteins

#Create dictionary:
lists = {
    'cel': ids_cel,
    'chi': ids_chi,
    'sta': ids_sta,
    'PET': ids_PET,
    'LDPE': ids_LDPE,
    'PS': ids_PS
}

id_map = defaultdict(list)

for name, lst in lists.items():
    for x in lst:
        id_map[x].append(name)



In [9]:
# IDs associated with specific substrates
def fasta_per_substrate(sub):
    cel_ids = lists[sub]
    
    # Filter dataframe
    df_cel = df[df['Protein ID'].isin(cel_ids)]
    
    
    with open("Bioinformatics_fasta/per_substrate_files/"f"{sub}" +" _sequences.fasta", "w") as f:
        for _, row in df_cel.iterrows():
            seq = row['Sequence'].removesuffix('**')
    
            f.write(f">{row['Protein ID']}\n")
            f.write(f"{seq}\n")

subs = ['cel', 'chi', 'sta', 'PET', 'LDPE', 'PS']

#save fastas with proteins binding to each substrate (for sequence alignment)
for sub in subs:
    fasta_per_substrate(sub)

In [10]:
#IDs present in both cellulose and PET lists


# Find IDs present in both groups
shared_ids = set(lists['cel']) & set(lists['PET'])

# Filter dataframe
df_shared = df[df['Protein ID'].isin(shared_ids)]

# Write FASTA file
with open("Bioinformatics_fasta/specific_promiscuous_files/cel_PET_shared.fasta", "w") as f:
    for _, row in df_shared.iterrows():
        seq = row['Sequence'].rstrip('*')  # remove trailing *

        f.write(f">{row['Protein ID']}\n")
        f.write(f"{seq}\n")

len(shared_ids)

123

In [11]:
# Group by how many lists they appear in
grouped = defaultdict(dict)
for k, v in id_map.items():
    grouped[len(v)][k] = v

one_sub = grouped[1]       # IDs of CBMs specifically binding to only one substrate
two_sub = grouped[2]
three_sub = grouped[3]
four_sub = grouped[4]
five_sub = grouped[5]
all_sub = grouped[6]       # Completely promiscuous CBMs

print('number of proteins with one substrate: ' + str(len(one_sub)))
print('number of proteins with two substrates: ' + str(len(two_sub)))
print('number of proteins with three substrates: ' + str(len(three_sub)))
print('number of proteins with four substrates: ' + str(len(four_sub)))
print('number of proteins with five substrates: ' + str(len(five_sub)))
print('number of proteins with binding all 6 substrates: ' + str(len(all_sub)))




number of proteins with one substrate: 123
number of proteins with two substrates: 153
number of proteins with three substrates: 114
number of proteins with four substrates: 80
number of proteins with five substrates: 47
number of proteins with binding all 6 substrates: 12


In [12]:
# For completely specific ones (grouped[1]), count how many are specific to each substrate
counts = Counter(v[0] for v in grouped[1].values())

print('amount of cellulose-specific: ' + str(counts['cel']))
print('amount of chitin-specific: ' + str(counts['chi']))
print('amount of starch-specific: ' + str(counts['sta']))
print('amount of PET-specific: ' + str(counts['PET']))
print('amount of LDPE-specific: ' + str(counts['LDPE']))
print('amount of PS-specific: ' + str(counts['PS']))


amount of cellulose-specific: 21
amount of chitin-specific: 37
amount of starch-specific: 21
amount of PET-specific: 6
amount of LDPE-specific: 5
amount of PS-specific: 33


In [13]:
# IDs for highly specific CBMs

    
def fasta_specific(sub):
    cel_only_ids = [
        protein_id
        for protein_id, substrates in one_sub.items()
        if substrates == [sub] or substrates == {sub}
    ]
    
    # Filter dataframe
    df_cel_only = df[df['Protein ID'].isin(cel_only_ids)]
    
    # Write FASTA file
    with open("Bioinformatics_fasta/specific_promiscuous_files/"f"{sub}" +" _specific_sequences.fasta", "w") as f:
        for _, row in df_cel_only.iterrows():
            seq = row['Sequence'].rstrip('*')
    
            f.write(f">{row['Protein ID']}\n")
            f.write(f"{seq}\n")

subs = ['cel', 'chi', 'sta', 'PET', 'LDPE', 'PS']

#save fastas with proteins binding to each substrate (for sequence alignment)
for sub in subs:
    fasta_specific(sub)